In [1]:
# %% 셀 1: avg_density 분위수(quintile) 기반 층화 후 10% 샘플링
import os
import re
import json
import math
import glob
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

EVAL_JSONL   = "./data/7_qwen_dataset_eval.jsonl"
INST_DIR     = "./data/6_telop_instances"

SAMPLE_RATIO = 0.1
SEED         = 42
N_BUCKETS    = 5  # quintile


# -------------------------
# 1) eval.jsonl에서 평가 대상 채널 목록 추출
# -------------------------
def extract_user_text(msg_content) -> str:
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(
            part.get("text", "") for part in msg_content if isinstance(part, dict)
        )
    return ""


eval_channels: set[str] = set()
with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    lines = f.readlines()

for line in tqdm(lines, desc="eval.jsonl 파싱"):
    ex = json.loads(line)
    user_msg = next(m for m in ex["messages"] if m["role"] == "user")
    text = extract_user_text(user_msg["content"])
    m_ch = re.search(r"채널:\s*([^\n]+)", text)
    if m_ch:
        eval_channels.add(m_ch.group(1).strip())

print(f"\neval.jsonl의 고유 채널 수: {len(eval_channels)}")


# -------------------------
# 2) 채널별 통계 feature 계산 (JSON 기반)
#    - 층화 기준: avg_density (영상 1초당 평균 동시 표시 텔롭 수)
#    - 나머지는 참고용
# -------------------------
def channel_stats(channel: str) -> dict | None:
    ch_dir = os.path.join(INST_DIR, channel)
    if not os.path.isdir(ch_dir):
        return None

    json_paths = sorted(glob.glob(os.path.join(glob.escape(ch_dir), "*.json")))
    if not json_paths:
        return None

    n_videos    = 0
    n_instances = 0
    dur_sum     = 0.0
    textlen_sum = 0
    density_sum = 0.0

    for jp in json_paths:
        try:
            with open(jp, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception:
            continue

        instances = data.get("instances", [])
        n_videos += 1

        if not instances:
            continue

        starts = np.array([inst["start_sec"] for inst in instances], dtype=np.float64)
        ends   = np.array([inst["end_sec"]   for inst in instances], dtype=np.float64)
        texts  = [inst["text"] for inst in instances]

        durs    = np.clip(ends - starts, 0, None)
        vid_len = float(ends.max()) if len(ends) else 0.0

        n_instances += len(instances)
        dur_sum     += float(durs.sum())
        textlen_sum += sum(len(t) for t in texts)

        if vid_len > 0:
            density_sum += float(durs.sum()) / vid_len

    if n_videos == 0:
        return None

    return {
        "n_videos":     n_videos,
        "n_instances":  n_instances,
        "avg_count":    n_instances / n_videos,
        "avg_duration": (dur_sum / n_instances) if n_instances > 0 else 0.0,
        "avg_text_len": (textlen_sum / n_instances) if n_instances > 0 else 0.0,
        "avg_density":  density_sum / n_videos,
    }


stats_rows = []
skipped = []
for ch in tqdm(sorted(eval_channels), desc="채널 통계 계산"):
    s = channel_stats(ch)
    if s is None:
        skipped.append(ch)
        continue
    stats_rows.append({"channel": ch, **s})

stats_df = pd.DataFrame(stats_rows)
print(f"\n✅ 통계 계산 완료: {len(stats_df)}개 채널 (skip {len(skipped)}개)")
print(stats_df.describe().T[["mean", "std", "min", "max"]])


# -------------------------
# 3) avg_density 분위수(quintile) 기반 bucket 할당
# -------------------------
stats_df = stats_df.sort_values("avg_density").reset_index(drop=True)
stats_df["bucket"] = pd.qcut(
    stats_df["avg_density"],
    q=N_BUCKETS,
    labels=list(range(N_BUCKETS)),
).astype(int)

print(f"\n📊 bucket별 avg_density 경계:")
bucket_stats = stats_df.groupby("bucket")["avg_density"].agg(["count", "min", "max", "mean"])
print(bucket_stats)


# -------------------------
# 4) bucket별 10% 샘플링 (올림, 최소 1개)
# -------------------------
random.seed(SEED)

sampled_channels: list[str] = []
bucket_to_sampled: dict[int, list[str]] = {}

for bid in tqdm(sorted(stats_df["bucket"].unique().tolist()), desc="bucket별 샘플링"):
    members = stats_df.loc[stats_df["bucket"] == bid, "channel"].tolist()
    members_sorted = sorted(members)
    n_pick = max(1, math.ceil(len(members_sorted) * SAMPLE_RATIO))
    picked = sorted(random.sample(members_sorted, n_pick))
    bucket_to_sampled[bid] = picked
    sampled_channels.extend(picked)

sampled_channels = sorted(set(sampled_channels))

print(f"\n📊 샘플링 결과")
print(f"  전체 후보 채널: {len(stats_df)}")
print(f"  샘플링된 채널: {len(sampled_channels)} ({len(sampled_channels)/len(stats_df):.1%})")
for bid, chs in bucket_to_sampled.items():
    total = int((stats_df["bucket"] == bid).sum())
    dens_min = stats_df.loc[stats_df["bucket"] == bid, "avg_density"].min()
    dens_max = stats_df.loc[stats_df["bucket"] == bid, "avg_density"].max()
    print(f"  bucket {bid}: {len(chs)}/{total}  (density {dens_min:.2f} ~ {dens_max:.2f})")

print(f"\n✅ 다음 셀에서 사용할 변수: sampled_channels, bucket_to_sampled, stats_df")

/home/taeyoung/miniconda3/envs/annotation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
eval.jsonl 파싱: 100%|██████████| 3320/3320 [00:00<00:00, 65159.72it/s]



eval.jsonl의 고유 채널 수: 664


채널 통계 계산: 100%|██████████| 664/664 [00:10<00:00, 61.95it/s]



✅ 통계 계산 완료: 664개 채널 (skip 0개)
                     mean          std         min           max
n_videos        99.987952     0.122219   98.000000    100.000000
n_instances   5633.626506  5885.766879  114.000000  53740.000000
avg_count       56.347965    58.886353    1.140000    537.400000
avg_duration     2.658320     2.426049    0.114035     20.649753
avg_text_len     6.440470     2.136661    1.519236     16.794069
avg_density      2.826470     2.239229    0.011535     16.820411

📊 bucket별 avg_density 경계:
        count       min        max      mean
bucket                                      
0         133  0.011535   0.664213  0.294376
1         133  0.685286   1.688121  1.160495
2         132  1.689540   3.240800  2.520000
3         133  3.257812   4.646314  3.939302
4         133  4.648890  16.820411  6.215873


bucket별 샘플링: 100%|██████████| 5/5 [00:00<00:00, 3755.64it/s]


📊 샘플링 결과
  전체 후보 채널: 664
  샘플링된 채널: 70 (10.5%)
  bucket 0: 14/133  (density 0.01 ~ 0.66)
  bucket 1: 14/133  (density 0.69 ~ 1.69)
  bucket 2: 14/132  (density 1.69 ~ 3.24)
  bucket 3: 14/133  (density 3.26 ~ 4.65)
  bucket 4: 14/133  (density 4.65 ~ 16.82)

✅ 다음 셀에서 사용할 변수: sampled_channels, bucket_to_sampled, stats_df


In [2]:
# %% 셀 2a: eval.jsonl의 prompt 토큰 수 측정
import json
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer

EVAL_JSONL  = "./data/7_qwen_dataset_eval.jsonl"
MODEL_PATH  = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep3-merged"
GEN_MAX_TOK = 8192  # 생성할 때 쓸 max_tokens
SAFETY_PAD  = 1024  # 안전 여유분


# -------------------------
# 1) 토크나이저 로드 (merged 모델 기준 - 학습/추론 동일)
# -------------------------
print(f"📦 토크나이저 로딩: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=False)


# -------------------------
# 2) 각 eval 샘플의 prompt 토큰 수 계산
#    - prompt = system + user, assistant 시작 마커까지
# -------------------------
with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    eval_lines = f.readlines()
print(f"📂 eval.jsonl 샘플 수: {len(eval_lines)}")

prompt_token_counts = []
completion_token_counts = []

for line in tqdm(eval_lines, desc="토큰 수 계산"):
    ex = json.loads(line)
    msgs = ex["messages"]
    prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
    asst_msgs   = [m for m in msgs if m["role"] == "assistant"]

    prompt_text = tokenizer.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    p_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    prompt_token_counts.append(len(p_ids))

    if asst_msgs:
        c_text = asst_msgs[0]["content"] + "<|im_end|>\n"
        c_ids = tokenizer.encode(c_text, add_special_tokens=False)
        completion_token_counts.append(len(c_ids))


# -------------------------
# 3) 통계 출력
# -------------------------
p = np.array(prompt_token_counts)
c = np.array(completion_token_counts)

print(f"\n📊 Prompt (system + user) 토큰 수")
print(f"  평균:      {p.mean():>8.0f}")
print(f"  중앙값:    {np.median(p):>8.0f}")
print(f"  p90:       {np.percentile(p, 90):>8.0f}")
print(f"  p95:       {np.percentile(p, 95):>8.0f}")
print(f"  p99:       {np.percentile(p, 99):>8.0f}")
print(f"  최소:      {p.min():>8.0f}")
print(f"  최대:      {p.max():>8.0f}")

print(f"\n📊 Completion (assistant 정답) 토큰 수")
print(f"  평균:      {c.mean():>8.0f}")
print(f"  중앙값:    {np.median(c):>8.0f}")
print(f"  p90:       {np.percentile(c, 90):>8.0f}")
print(f"  p95:       {np.percentile(c, 95):>8.0f}")
print(f"  p99:       {np.percentile(c, 99):>8.0f}")
print(f"  최소:      {c.min():>8.0f}")
print(f"  최대:      {c.max():>8.0f}")


# -------------------------
# 4) 권장 context-length 계산
#    = (prompt 최대) + (생성 max_tokens) + (안전 여유분)
# -------------------------
needed = int(p.max()) + GEN_MAX_TOK + SAFETY_PAD

# 2의 거듭제곱으로 올림 (sglang이 정렬된 크기를 선호)
power = 1
while power < needed:
    power *= 2
recommended = power

print(f"\n📐 context-length 계산")
print(f"  prompt 최대:       {p.max()}")
print(f"  + 생성 max_tokens: {GEN_MAX_TOK}")
print(f"  + 안전 여유분:     {SAFETY_PAD}")
print(f"  합계:              {needed}")
print(f"  → 권장 값:         {recommended} (2의 거듭제곱 올림)")

print(f"\n✅ 셀 2에서 --context-length {recommended} 사용")

📦 토크나이저 로딩: ./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep3-merged
📂 eval.jsonl 샘플 수: 3320


토큰 수 계산: 100%|██████████| 3320/3320 [00:05<00:00, 643.31it/s]


📊 Prompt (system + user) 토큰 수
  평균:           491
  중앙값:         434
  p90:            788
  p95:            906
  p99:           1303
  최소:           228
  최대:          2186

📊 Completion (assistant 정답) 토큰 수
  평균:          1306
  중앙값:         682
  p90:           3114
  p95:           4428
  p99:           9783
  최소:             6
  최대:         52150

📐 context-length 계산
  prompt 최대:       2186
  + 생성 max_tokens: 8192
  + 안전 여유분:     1024
  합계:              11402
  → 권장 값:         16384 (2의 거듭제곱 올림)

✅ 셀 2에서 --context-length 16384 사용


In [3]:
# %% 셀 2: 기존 sglang 서버 종료 + ep3 merged 모델 서버 시작
import os
import subprocess
import time
import requests
import signal

os.environ["HF_HOME"] = "/home/taeyoung/nfs-mount/chi2027/.cache/huggingface"

MODEL_PATH      = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep3-merged"
PORT            = 8000
SERVER_LOG      = "sglang_server.log"
CONTEXT_LENGTH  = 131072
#CONTEXT_LENGTH  = 16384

# -------------------------
# 1) 기존 서버 프로세스 종료
# -------------------------
print("🛑 기존 sglang 서버 종료 시도...")

# (a) 같은 커널에서 띄운 핸들이 있으면 종료
if "server_process" in globals() and server_process.poll() is None:
    try:
        server_process.terminate()
        server_process.wait(timeout=10)
        print("   ✅ 이전 server_process 핸들로 종료")
    except Exception as e:
        try:
            server_process.kill()
            print(f"   ⚠️ terminate 실패 → kill ({e})")
        except Exception:
            pass

# (b) 포트 점유 프로세스 정리
try:
    out = subprocess.run(
        ["lsof", "-t", f"-i:{PORT}"],
        capture_output=True, text=True, timeout=5,
    ).stdout.strip()
    pids = [int(p) for p in out.splitlines() if p.strip().isdigit()]
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
            print(f"   ✅ PID {pid} SIGTERM")
        except ProcessLookupError:
            pass
    time.sleep(3)
    out2 = subprocess.run(
        ["lsof", "-t", f"-i:{PORT}"],
        capture_output=True, text=True, timeout=5,
    ).stdout.strip()
    for p in out2.splitlines():
        if p.strip().isdigit():
            try:
                os.kill(int(p), signal.SIGKILL)
                print(f"   ✅ PID {p} SIGKILL")
            except ProcessLookupError:
                pass
except FileNotFoundError:
    print("   ⚠️ lsof 없음 — 포트 점유 확인 skip")

# (c) 포트 비워질 때까지 대기
for _ in range(10):
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=1)
        if r.status_code == 200:
            time.sleep(1)
            continue
    except Exception:
        break
print("   ✅ 포트 정리 완료\n")


# -------------------------
# 2) ep3 merged 모델로 새 서버 시작
# -------------------------
assert os.path.isdir(MODEL_PATH), f"모델 경로 없음: {MODEL_PATH}"

sglang_log = open(SERVER_LOG, "w")

server_process = subprocess.Popen(
    [
        "python", "-m", "sglang.launch_server",
        "--model-path", MODEL_PATH,
        "--port", str(PORT),
        "--mem-fraction-static", "0.9",
        "--context-length", str(CONTEXT_LENGTH),
        "--reasoning-parser", "qwen3",
        "--attention-backend", "triton",
    ],
    stdout=sglang_log,
    stderr=subprocess.STDOUT,
)

print(f"⏳ SGLang 서버 시작 중...")
print(f"   모델: {MODEL_PATH}")
print(f"   context-length: {CONTEXT_LENGTH}")

ready = False
for i in range(600):
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=1)
        if r.status_code == 200:
            print(f"✅ 서버 준비 완료! ({i}초)")
            ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    print(f"❌ 서버 시작 실패. 로그 확인: tail -n 200 {SERVER_LOG}")

🛑 기존 sglang 서버 종료 시도...
   ✅ 포트 정리 완료

⏳ SGLang 서버 시작 중...
   모델: ./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep3-merged
   context-length: 131072
✅ 서버 준비 완료! (20초)


In [ ]:
# %% 셀 3: 70채널 × held-out 영상 × 70채널 swap inference
import os
import re
import json
import time
import httpx
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

EVAL_JSONL = "./data/7_qwen_dataset_eval.jsonl"
OUTPUT_DIR = "./data/7_qwen_inference"
MODEL_NAME = "./model/qwen3.5-9b-lora-teloptextgen-rank64-split-ep3-merged"

TEMPERATURE = 0.1
TOP_P       = 1.0
MAX_TOKENS  = 126976
#MAX_TOKENS  = 14000

MAX_WORKERS = 50
TIMEOUT     = 600

os.makedirs(OUTPUT_DIR, exist_ok=True)

client = OpenAI(
    base_url=f"http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=httpx.Timeout(TIMEOUT, connect=5.0),
)


# -------------------------
# 1) eval.jsonl에서 "70개 샘플 채널에 속한 영상들"만 필터링
# -------------------------
def parse_user_text(msg_content) -> str:
    if isinstance(msg_content, str):
        return msg_content
    if isinstance(msg_content, list):
        return "\n".join(
            part.get("text", "") for part in msg_content if isinstance(part, dict)
        )
    return ""


sampled_set = set(sampled_channels)

with open(EVAL_JSONL, "r", encoding="utf-8") as f:
    eval_lines = f.readlines()

target_entries = []
for line in tqdm(eval_lines, desc="eval.jsonl 필터링"):
    ex = json.loads(line)
    msgs = ex["messages"]
    sys_msg  = next(m for m in msgs if m["role"] == "system")
    user_msg = next(m for m in msgs if m["role"] == "user")
    user_text = parse_user_text(user_msg["content"])

    m_ch = re.search(r"채널:\s*([^\n]+)", user_text)
    m_vn = re.search(r"영상:\s*([^\n]+)", user_text)
    if not m_ch or not m_vn:
        continue

    channel = m_ch.group(1).strip()
    video   = m_vn.group(1).strip()

    if channel not in sampled_set:
        continue

    target_entries.append({
        "channel":    channel,
        "video_name": video,
        "system":     sys_msg["content"],
        "user_text":  user_text,
    })

print(f"\n✅ 대상 영상 수: {len(target_entries)}")
print(f"   → 총 생성 요청: {len(target_entries) * len(sampled_channels)}")


# -------------------------
# 2) 작업 리스트 구성 (이미 저장된 결과 skip)
# -------------------------
def swap_channel_line(user_text: str, new_channel: str) -> str:
    return re.sub(
        r"채널:\s*[^\n]+",
        f"채널: {new_channel}",
        user_text,
        count=1,
    )


def output_path(orig_ch: str, video: str, cond_ch: str) -> str:
    return os.path.join(OUTPUT_DIR, orig_ch, video, f"{cond_ch}.json")


jobs = []
for ent in tqdm(target_entries, desc="job 리스트 구성"):
    for cond_ch in sampled_channels:
        out_path = output_path(ent["channel"], ent["video_name"], cond_ch)
        if os.path.exists(out_path):
            continue
        jobs.append({
            "orig_channel": ent["channel"],
            "video_name":   ent["video_name"],
            "cond_channel": cond_ch,
            "system":       ent["system"],
            "user_text":    swap_channel_line(ent["user_text"], cond_ch),
            "out_path":     out_path,
        })

print(f"\n📋 실행할 job: {len(jobs)} "
      f"(skip: {len(target_entries) * len(sampled_channels) - len(jobs)})")


# -------------------------
# 3) 단일 요청 처리
# -------------------------
def call_model(job: dict) -> dict:
    t0 = time.time()
    try:
        resp = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": job["system"]},
                {"role": "user",   "content": job["user_text"]},
            ],
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_tokens=MAX_TOKENS,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        elapsed = time.time() - t0

        raw = resp.choices[0].message.content.strip()
        cleaned = raw
        if cleaned.startswith("```"):
            cleaned = re.sub(r"^```[a-zA-Z]*\n?", "", cleaned)
            cleaned = re.sub(r"\n?```$", "", cleaned)

        parsed = json.loads(cleaned)

        os.makedirs(os.path.dirname(job["out_path"]), exist_ok=True)
        record = {
            "orig_channel":  job["orig_channel"],
            "video_name":    job["video_name"],
            "cond_channel":  job["cond_channel"],
            "model":         MODEL_NAME,
            "params": {
                "temperature": TEMPERATURE,
                "top_p":       TOP_P,
                "max_tokens":  MAX_TOKENS,
            },
            "system":        job["system"],
            "user_text":     job["user_text"],
            "raw_output":    raw,
            "instances":     parsed.get("instances", []) if isinstance(parsed, dict) else parsed,
            "elapsed_sec":   round(elapsed, 2),
            "prompt_tokens":     getattr(resp.usage, "prompt_tokens", None),
            "completion_tokens": getattr(resp.usage, "completion_tokens", None),
        }
        with open(job["out_path"], "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)

        return {"success": True, "elapsed": elapsed}

    except json.JSONDecodeError as e:
        return {"success": False, "error": f"JSONDecodeError: {str(e)[:100]}",
                "elapsed": time.time() - t0}
    except Exception as e:
        return {"success": False, "error": str(e)[:200],
                "elapsed": time.time() - t0}


# -------------------------
# 4) 병렬 실행
# -------------------------
if not jobs:
    print("\n✅ 모든 job이 이미 완료 — 새로 생성할 것 없음")
else:
    n_success = 0
    n_fail    = 0
    errors    = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(call_model, j): j for j in jobs}

        with tqdm(total=len(jobs), desc="inference") as pbar:
            for fut in as_completed(futures):
                job = futures[fut]
                res = fut.result()
                if res["success"]:
                    n_success += 1
                else:
                    n_fail += 1
                    errors.append({
                        "orig":  job["orig_channel"],
                        "video": job["video_name"],
                        "cond":  job["cond_channel"],
                        "err":   res["error"],
                    })
                pbar.update(1)
                pbar.set_postfix(ok=n_success, fail=n_fail)

    print(f"\n📊 완료")
    print(f"  성공: {n_success}")
    print(f"  실패: {n_fail}")
    if errors[:5]:
        print(f"\n실패 샘플:")
        for e in errors[:5]:
            print(f"  [{e['orig']}] {e['video']} ← {e['cond']}: {e['err']}")

eval.jsonl 필터링: 100%|██████████| 3320/3320 [00:00<00:00, 63276.30it/s]



✅ 대상 영상 수: 350
   → 총 생성 요청: 24500


job 리스트 구성: 100%|██████████| 350/350 [00:00<00:00, 402.31it/s]



📋 실행할 job: 14793 (skip: 9707)


inference:   2%|▏         | 295/14793 [2:42:22<156:16:51, 38.81s/it, fail=229, ok=66] 